# Single Gaussian Beam

Build one Gaussian beam, propagate it through a thin lens, and evaluate the field on a detector grid. The notebook also compares the simple Python-loop evaluator with the batched JAX scan evaluator so API changes are easy to catch.


In [ ]:
import os
os.environ["JAX_ENABLE_X64"] = "1"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.2"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp

from temgym_core.components import Detector, Lens
from temgym_core.evaluate import evaluate_gaussians_for, evaluate_gaussians_jax_scan
from temgym_core.gaussian import make_gaussian
from temgym_core.run import run_to_end

jax.config.update("jax_enable_x64", True)


## Geometry

The grid is intentionally small enough to run quickly on CPU while still showing the Gaussian envelope and lens phase curvature.


In [ ]:
window = 120e-9
shape = (192, 192)
pixel = window / shape[0]
input_grid = Detector(z=0.0, pixel_size=(pixel, pixel), shape=shape)
detector = Detector(z=20e-6, pixel_size=(pixel, pixel), shape=shape)

beam = make_gaussian(
    x=0.0,
    y=0.0,
    dx=0.0,
    dy=0.0,
    voltage=200e3,
    waist_x=10e-9,
    waist_y=10e-9,
)
lens = Lens(z=0.0, focal_length=40e-6)
out = run_to_end(beam, (lens, detector))


## Evaluate

`evaluate_gaussians_for` is easy to read and `evaluate_gaussians_jax_scan` is the scalable path. They should agree for the same beam bundle.


In [ ]:
field_loop = evaluate_gaussians_for(out, detector)
field_scan = evaluate_gaussians_jax_scan(out, detector, batch_size=1)
np.testing.assert_allclose(np.asarray(field_scan), np.asarray(field_loop), rtol=1e-7, atol=1e-12)
field = np.asarray(field_scan)


## Visualise

The amplitude shows the Gaussian envelope; the phase shows the local wavefront curvature after the lens and propagation.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

im0 = ax[0].imshow(np.abs(field), extent=detector.extent, origin="lower", cmap="inferno")
ax[0].set_title("Amplitude")
ax[0].set_xlabel("x (m)")
ax[0].set_ylabel("y (m)")
fig.colorbar(im0, ax=ax[0])

im1 = ax[1].imshow(np.angle(field), extent=detector.extent, origin="lower", cmap="twilight")
ax[1].set_title("Phase")
ax[1].set_xlabel("x (m)")
ax[1].set_ylabel("y (m)")
fig.colorbar(im1, ax=ax[1])
